# Mathematical Derivation of the Attention Mechanism


## Setup

In [1]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

## Part A — Basic mathematical foundations
### Question 1, tasks 1–4: Given matrices and dimensions
There are three tokens with two features each. Multiplying a token's row by a projection matrix produces its Query, Key, or Value.

$$X=\begin{bmatrix}1&0\\0&1\\1&1\end{bmatrix}\in\mathbb{R}^{3\times2}$$
$$W_Q=W_K=\begin{bmatrix}1&0\\0&1\end{bmatrix}\in\mathbb{R}^{2\times2},\qquad W_V=\begin{bmatrix}2&0\\0&2\end{bmatrix}\in\mathbb{R}^{2\times2}.$$
For any product $B=XM$, its entry is $B_{ij}=\sum_{r=1}^{2}X_{ir}M_{rj}$. Thus each result has shape $(3\times2)(2\times2)=3\times2$.

In [2]:
X = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
W_Q = np.array([[1, 0], [0, 1]], dtype=float)
W_K = np.array([[1, 0], [0, 1]], dtype=float)
W_V = np.array([[2, 0], [0, 2]], dtype=float)
print("X:", X.shape)
print("W_Q, W_K, W_V:", W_Q.shape, W_K.shape, W_V.shape)

X: (3, 2)
W_Q, W_K, W_V: (2, 2) (2, 2) (2, 2)


### 1. Derive $Q=XW_Q$
Each Query is a row of $X$ multiplied by $W_Q$. Since $W_Q$ is the identity, the Query vectors equal the input embeddings.

$$Q=\begin{bmatrix}
1(1)+0(0)&1(0)+0(1)\\
0(1)+1(0)&0(0)+1(1)\\
1(1)+1(0)&1(0)+1(1)
\end{bmatrix}
=\begin{bmatrix}1&0\\0&1\\1&1\end{bmatrix}.$$
$$\underbrace{X}_{3\times2}\underbrace{W_Q}_{2\times2}=\underbrace{Q}_{3\times2}.$$

In [3]:
Q = X @ W_Q
print("Q =\n", Q)
print("Q shape:", Q.shape)

Q =
 [[1. 0.]
 [0. 1.]
 [1. 1.]]
Q shape: (3, 2)


### 2. Derive $K=XW_K$
The same row-by-column calculation produces the Keys. Here $W_K$ is also the identity, so $K=Q=X$; learned Query and Key projections need not be equal in a real model.

$$K=\begin{bmatrix}
1(1)+0(0)&1(0)+0(1)\\
0(1)+1(0)&0(0)+1(1)\\
1(1)+1(0)&1(0)+1(1)
\end{bmatrix}
=\begin{bmatrix}1&0\\0&1\\1&1\end{bmatrix}.$$
$$\underbrace{X}_{3\times2}\underbrace{W_K}_{2\times2}=\underbrace{K}_{3\times2}.$$

In [4]:
K = X @ W_K
print("K =\n", K)
print("K shape:", K.shape)

K =
 [[1. 0.]
 [0. 1.]
 [1. 1.]]
K shape: (3, 2)


### 3. Derive $V=XW_V$
The Value projection doubles both coordinates of each embedding. These are the vectors that will be combined after the attention weights are calculated.

$$V=\begin{bmatrix}
1(2)+0(0)&1(0)+0(2)\\
0(2)+1(0)&0(0)+1(2)\\
1(2)+1(0)&1(0)+1(2)
\end{bmatrix}
=\begin{bmatrix}2&0\\0&2\\2&2\end{bmatrix}.$$
$$\underbrace{X}_{3\times2}\underbrace{W_V}_{2\times2}=\underbrace{V}_{3\times2}.$$

In [5]:
V = X @ W_V
print("V =\n", V)
print("V shape:", V.shape)

V =
 [[2. 0.]
 [0. 2.]
 [2. 2.]]
V shape: (3, 2)


## Part B — Deriving attention scores
### Question 2, task 5: Transpose $K$
Transposing makes each Key a column, so each Query row can be compared with every Key. The transpose changes the shape from $3\times2$ to $2\times3$.

$$K=\begin{bmatrix}1&0\\0&1\\1&1\end{bmatrix},\qquad K^T=\begin{bmatrix}1&0&1\\0&1&1\end{bmatrix}.$$

In [6]:
K_T = K.T
print("K transpose =\n", K_T)
print("Shape:", K_T.shape)

K transpose =
 [[1. 0. 1.]
 [0. 1. 1.]]
Shape: (2, 3)


### Task 6: Calculate every entry of $QK^T$
Entry $(i,j)$ is the dot product between Query $i$ and Key $j$. There are three Queries and three Keys, so this gives nine scores.

$$QK^T=
\begin{bmatrix}
1(1)+0(0)&1(0)+0(1)&1(1)+0(1)\\
0(1)+1(0)&0(0)+1(1)&0(1)+1(1)\\
1(1)+1(0)&1(0)+1(1)&1(1)+1(1)
\end{bmatrix}
=\begin{bmatrix}1&0&1\\0&1&1\\1&1&2\end{bmatrix}.$$
$$\underbrace{Q}_{3\times2}\underbrace{K^T}_{2\times3}=\underbrace{QK^T}_{3\times3}.$$

In [7]:
scores = Q @ K_T
print("QK^T =\n", scores)
print("Score shape:", scores.shape)

QK^T =
 [[1. 0. 1.]
 [0. 1. 1.]
 [1. 1. 2.]]
Score shape: (3, 3)


### Tasks 7–9: Find $d_k$ and scale the scores
Each Key has two components, so $d_k=2$ and $\sqrt{d_k}=\sqrt2\approx1.414214$. Divide every score by this scalar; the matrix dimensions stay $3\times3$.

$$S=\frac{QK^T}{\sqrt2}=
\begin{bmatrix}
1/\sqrt2&0/\sqrt2&1/\sqrt2\\
0/\sqrt2&1/\sqrt2&1/\sqrt2\\
1/\sqrt2&1/\sqrt2&2/\sqrt2
\end{bmatrix}
\approx\begin{bmatrix}
0.707107&0&0.707107\\
0&0.707107&0.707107\\
0.707107&0.707107&1.414214
\end{bmatrix}.$$

In [8]:
d_k = K.shape[1]
scale = np.sqrt(d_k)
scaled_scores = scores / scale
print("d_k:", d_k, "sqrt(d_k):", scale)
print("Scaled scores =\n", scaled_scores)

d_k: 2 sqrt(d_k): 1.4142135623730951
Scaled scores =
 [[0.707107 0.       0.707107]
 [0.       0.707107 0.707107]
 [0.707107 0.707107 1.414214]]


## Part C — Softmax derivation
### Question 3, tasks 10–12: Work through $z=[2,1,0]$
Exponentiation makes every numerator positive. Dividing by the common sum turns the three values into a normalized distribution.

$$e^2\approx7.389056,\qquad e^1\approx2.718282,\qquad e^0=1.$$
$$D=e^2+e^1+e^0\approx7.389056+2.718282+1=11.107338.$$
$$P_1=\frac{e^2}{D}=\frac{7.389056}{11.107338}\approx0.665241,$$
$$P_2=\frac{e^1}{D}=\frac{2.718282}{11.107338}\approx0.244728,$$
$$P_3=\frac{e^0}{D}=\frac{1}{11.107338}\approx0.090031.$$
The input and output are each a vector of length 3; the denominator is a scalar.

In [9]:
z = np.array([2, 1, 0], dtype=float)
exp_z = np.exp(z)
denominator = exp_z.sum()
print("Exponentials:", exp_z)
print("Denominator:", denominator)

Exponentials: [7.389056 2.718282 1.      ]
Denominator: 11.107337927389695


In [10]:
probabilities = exp_z / denominator
for i, probability in enumerate(probabilities, start=1):
    print(f"P{i} = {probability:.6f}")

P1 = 0.665241
P2 = 0.244728
P3 = 0.090031


### Task 13: Verify the sum
The sum is exactly one algebraically, not just approximately one after rounding. This is the normalization property used independently for each Query's attention row.

$$P_1+P_2+P_3=\frac{e^2+e^1+e^0}{e^2+e^1+e^0}=1.$$
$$0.665241+0.244728+0.090031=1.000000\quad\text{(rounded values).}$$

In [11]:
print("Probability sum:", probabilities.sum())
assert np.isclose(probabilities.sum(), 1)

Probability sum: 1.0


### Question 4: Why softmax gives attention weights
For a row of finite real scores $s_1,\ldots,s_n$, define $a_j=e^{s_j}/\sum_{r=1}^{n}e^{s_r}$. The denominator is positive, every $a_j>0$, and $\sum_j a_j=1$, so the vector is a valid probability distribution over Keys.

$$\frac{a_j}{a_l}=e^{s_j-s_l}.$$
A larger score receives a larger weight, and score differences control relative weight. These normalized weights represent the model's allocation of attention, not a guaranteed probability that a token is linguistically relevant.

**Stable computation:** Subtracting the largest score leaves the result unchanged and prevents very large positive exponentials.

$$\frac{e^{s_j-m}}{\sum_r e^{s_r-m}}=\frac{e^{-m}e^{s_j}}{e^{-m}\sum_r e^{s_r}}=\frac{e^{s_j}}{\sum_r e^{s_r}},\qquad m=\max_r s_r.$$

## Part D — Complete attention calculation
### 1. Apply softmax row by row
Part C's vector was a separate example. Here softmax must be applied to the actual rows of $S$, with each row normalized across its three Keys.

Set $t=1/\sqrt2$, $u=e^t\approx2.028115$, and $v=e^{2t}\approx4.113250$. The exponentials below are elementwise, not a matrix exponential.

$$\exp(S)=\begin{bmatrix}u&1&u\\1&u&u\\u&u&v\end{bmatrix}\in\mathbb R^{3\times3}.$$
$$D_1=D_2=2u+1\approx5.056230,\qquad D_3=2u+v\approx8.169480.$$

In [12]:
exp_scores = np.exp(scaled_scores)
row_totals = exp_scores.sum(axis=1, keepdims=True)
print("Elementwise exponentials =\n", exp_scores)
print("Row denominators =\n", row_totals)
print("Shapes:", exp_scores.shape, row_totals.shape)

Elementwise exponentials =
 [[2.028115 1.       2.028115]
 [1.       2.028115 2.028115]
 [2.028115 2.028115 4.11325 ]]
Row denominators =
 [[5.05623]
 [5.05623]
 [8.16948]]
Shapes: (3, 3) (3, 1)


### 2. Calculate all nine attention weights
Each entry in a row is divided by that row's denominator. The attention matrix keeps the shape $3\times3$; $A_{ij}$ tells how much Query $i$ uses Value $j$.

$$A=\begin{bmatrix}
\frac{2.028115}{5.056230}&\frac{1}{5.056230}&\frac{2.028115}{5.056230}\\
\frac{1}{5.056230}&\frac{2.028115}{5.056230}&\frac{2.028115}{5.056230}\\
\frac{2.028115}{8.169480}&\frac{2.028115}{8.169480}&\frac{4.113250}{8.169480}
\end{bmatrix}.$$
Using full-precision values before rounding gives the **final attention matrix**:

$$\boxed{A=\operatorname{softmax}_{\rm row}(QK^T/\sqrt{d_k})\approx
\begin{bmatrix}
0.401112&0.197776&0.401112\\
0.197776&0.401112&0.401112\\
0.248255&0.248255&0.503490
\end{bmatrix}}.$$
The first two rows each sum to $0.401112+0.197776+0.401112=1$; the last sums to $0.248255+0.248255+0.503490=1$.

In [13]:
A = exp_scores / row_totals
print("Final attention matrix A =\n", A)
print("Row sums:", A.sum(axis=1))
assert np.allclose(A.sum(axis=1), 1)

Final attention matrix A =
 [[0.401112 0.197776 0.401112]
 [0.197776 0.401112 0.401112]
 [0.248255 0.248255 0.50349 ]]
Row sums: [1. 1. 1.]


### 3. Verify using a small stable softmax function
The manual calculation above is complete. This function implements the same definition with row-wise maximum subtraction and does not call a library softmax or attention routine.

In [14]:
def softmax_rows(matrix):
    shifted = matrix - matrix.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=1, keepdims=True)
A_stable = softmax_rows(scaled_scores)
assert np.allclose(A, A_stable)
print("Direct and stable softmax agree:", np.allclose(A, A_stable))

Direct and stable softmax agree: True


### 4. Multiply the attention matrix by $V$
Each output row is a weighted sum of the three Value rows: $o_i=A_{i1}v_1+A_{i2}v_2+A_{i3}v_3$. The inner dimension of 3 matches, leaving two output features for each of three tokens.

$$\underbrace{A}_{3\times3}\underbrace{V}_{3\times2}=\underbrace{O}_{3\times2}.$$
$$O=\begin{bmatrix}
2A_{11}+0A_{12}+2A_{13}&0A_{11}+2A_{12}+2A_{13}\\
2A_{21}+0A_{22}+2A_{23}&0A_{21}+2A_{22}+2A_{23}\\
2A_{31}+0A_{32}+2A_{33}&0A_{31}+2A_{32}+2A_{33}
\end{bmatrix}.$$
Using displayed weights for the substitutions below (and full precision for the final rounded results):

$$O_{11}=2(0.401112)+0(0.197776)+2(0.401112)\approx1.604448,$$
$$O_{12}=0(0.401112)+2(0.197776)+2(0.401112)\approx1.197776,$$
$$O_{21}=2(0.197776)+0(0.401112)+2(0.401112)\approx1.197776,$$
$$O_{22}=0(0.197776)+2(0.401112)+2(0.401112)\approx1.604448,$$
$$O_{31}=2(0.248255)+0(0.248255)+2(0.503490)\approx1.503490,$$
$$O_{32}=0(0.248255)+2(0.248255)+2(0.503490)\approx1.503490.$$

The **final attention output** is:

$$\boxed{\operatorname{Output}=AV\approx\begin{bmatrix}
1.604448&1.197776\\
1.197776&1.604448\\
1.503490&1.503490
\end{bmatrix}}.$$

In [15]:
output = A @ V
print("Final attention output =\n", output)
print("Output shape:", output.shape)

Final attention output =
 [[1.604448 1.197776]
 [1.197776 1.604448]
 [1.50349  1.50349 ]]
Output shape: (3, 2)


### 5. Interpret the result and check it
Token 1 attends equally to tokens 1 and 3; token 2 attends equally to tokens 2 and 3. Token 3 gives the highest weight to itself because its self-dot-product is 2, compared with 1 for the other Keys.

Every row of $A$ is nonnegative and sums to one, so each output is a convex combination of the Value vectors. Here both output coordinates must lie between 0 and 2; attention is not simply selecting one Value.

In [16]:
expected_A = np.array([[0.401112, 0.197776, 0.401112], [0.197776, 0.401112, 0.401112], [0.248255, 0.248255, 0.503490]])
expected_output = np.array([[1.604448, 1.197776], [1.197776, 1.604448], [1.503490, 1.503490]])
assert np.allclose(A, expected_A, atol=1e-6, rtol=0)
assert np.allclose(output, expected_output, atol=1e-6, rtol=0)
assert np.all((output >= 0) & (output <= 2))
print("Manual attention weights and output match the numerical calculation.")

Manual attention weights and output match the numerical calculation.


## Part E — Why divide by $\sqrt{d_k}$?
### Question 5, tasks 14–15: Derive the dot-product variance
Use the assignment's independence assumption for all components: $q_i$ and $k_i$ are independent, and different component pairs are independent. Let $r_i=q_i k_i$, where each original component has mean 0 and variance 1.

**Step 1: Mean and second moment of a product.** Independence lets us factor expectations.

$$E[r_i]=E[q_i k_i]=E[q_i]E[k_i]=0,$$
$$E[q_i^2]=\operatorname{Var}(q_i)+(E[q_i])^2=1,\qquad E[k_i^2]=1,$$
$$E[r_i^2]=E[q_i^2k_i^2]=E[q_i^2]E[k_i^2]=1.$$

**Step 2: Variance of a product.** Subtracting the square of its mean gives unit variance.

$$\operatorname{Var}(r_i)=E[r_i^2]-(E[r_i])^2=1-0=1.$$

**Step 3: Variance of the sum.** Independence across component pairs makes all cross-covariances zero.

$$\operatorname{Var}(q\cdot k)=\operatorname{Var}\left(\sum_{i=1}^{d_k}r_i\right)
=\sum_{i=1}^{d_k}\operatorname{Var}(r_i)+2\sum_{i<j}\operatorname{Cov}(r_i,r_j)
=\sum_{i=1}^{d_k}1+0=\boxed{d_k}.$$
The unscaled dot product therefore has mean 0 and standard deviation $\sqrt{d_k}$. This is a statistical justification under the stated assumptions, not an exact description of every learned attention layer; in Parts A–D, $Q=K$, so Query and Key components are not independent random samples.

### Task 16: Scaling controls the variance
For a scalar $c$, $\operatorname{Var}(cZ)=c^2\operatorname{Var}(Z)$. Applying this to the dot product gives a score with variance 1, independent of the vector dimension under these assumptions.

$$\operatorname{Var}\left(\frac{q\cdot k}{\sqrt{d_k}}\right)
=\frac{1}{d_k}\operatorname{Var}(q\cdot k)=\frac{d_k}{d_k}=\boxed{1}.$$
Scaling controls typical magnitude; it does not strictly bound every score between $-1$ and $1$.

In [17]:
for dimension in [2, 8, 64, 512]:
    raw_variance = dimension
    scaled_variance = raw_variance / dimension
    print(f"d_k={dimension}: raw variance={raw_variance}, scaled variance={scaled_variance:.1f}")

d_k=2: raw variance=2, scaled variance=1.0
d_k=8: raw variance=8, scaled variance=1.0
d_k=64: raw variance=64, scaled variance=1.0
d_k=512: raw variance=512, scaled variance=1.0


### Task 17: Large unscaled scores and softmax saturation
As dimension increases, unscaled dot products have a larger typical spread, which can produce large gaps between scores and nearly one-hot softmax weights. The softmax Jacobian explains why gradients through these weights can then become small.

$$\frac{\partial a_i}{\partial s_j}=a_i(\delta_{ij}-a_j),$$
where $\delta_{ij}=1$ if $i=j$ and 0 otherwise. If one probability is near 1 and the others are near 0, the Jacobian entries approach 0, making changes to the attention allocation harder to learn.

A large common offset alone does **not** sharpen softmax, because softmax is shift-invariant; differences between logits matter. Maximum subtraction prevents numerical overflow, while division by $\sqrt{d_k}$ controls score spread—these solve different problems.

In [18]:
example_scores = np.array([[8.0, 0.0, -8.0]])
print("Without scaling:", softmax_rows(example_scores))
print("Scaled with sqrt(64):", softmax_rows(example_scores / np.sqrt(64)))

Without scaling: [[0.999665 0.000335 0.      ]]
Scaled with sqrt(64): [[0.665241 0.244728 0.090031]]


## Part F — Self-attention
### Question 6: What does “it” refer to?
**Sentence:** “The student opened the book because it was interesting.”

The intended referent is **the book**. In self-attention, Queries, Keys, and Values are projected from representations of tokens in the same sequence; positional information and representations from earlier layers also help provide context.

**Query for “it”.** If $x_{\text{it}}\in\mathbb R^{1\times d_{model}}$ is its current representation, then $q_{\text{it}}=x_{\text{it}}W_Q\in\mathbb R^{1\times d_k}$. This learned vector expresses the information needed to update “it”; it is not a hand-written search rule.

**Keys.** Each token has $k_j=x_jW_K\in\mathbb R^{1\times d_k}$, so $K\in\mathbb R^{n\times d_k}$. In unmasked self-attention, the Query for “it” can compare with every token, including itself; a causal decoder restricts this to available earlier tokens and itself.

**Desired high-weight Key.** A head capturing this relationship should give $k_{\text{book}}$ a high compatibility score:

$$s_{\text{it},j}=\frac{q_{\text{it}}k_j^T}{\sqrt{d_k}},\qquad
\alpha_{\text{it},\text{book}}=\frac{e^{s_{\text{it},\text{book}}}}{\sum_r e^{s_{\text{it},r}}}.$$
The dot product has dimensions $(1\times d_k)(d_k\times1)=1\times1$, so each score is a scalar. A model could learn from many examples that a book is something described as interesting, but this attention pattern is not guaranteed by the formula alone.

**Corresponding Value.** The Value $v_{\text{book}}=x_{\text{book}}W_V\in\mathbb R^{1\times d_v}$ carries information about the book into the updated representation of “it”. The complete output is a weighted sum, not just a copy of the single highest-weight Value:

$$o_{\text{it}}=\sum_{j=1}^n\alpha_{\text{it},j}v_j,
\qquad(1\times n)(n\times d_v)=1\times d_v.$$

**Connection to Transformers.** The full path is $X\rightarrow(Q,K,V)\rightarrow QK^T\rightarrow QK^T/\sqrt{d_k}\rightarrow A\rightarrow AV$. Transformer layers combine this attention output with output projection, residual connections, normalization, and feed-forward transformations; multiple layers can build richer contextual representations. A causal decoder's representation at “it” cannot yet use the later word “interesting”, while an encoder can use both directions.

## Bonus — Multi-head attention
### Question 7, task 18: Head dimensions
The bonus is a separate setup with $d_{model}=8$, not the two-feature numerical example above. With two equally sized heads and the standard choice $d_v=d_k$:

$$d_k=\frac{d_{model}}{h}=\frac82=\boxed4,\qquad d_v=4.$$

In [19]:
d_model = 8
h = 2
head_dimension = d_model // h
print("Dimensions per head:", head_dimension)

Dimensions per head: 4


### Task 19: Why heads can learn different relationships
Let the bonus inputs be $Q_b,K_b,V_b\in\mathbb R^{n\times8}$, using the subscript $b$ to distinguish them from the earlier $3\times2$ matrices. Each head has its own learned projections, so the compatibility function depends on a different matrix product.

$$s_{ab}^{(i)}=\frac{(q_aW_i^Q)(k_bW_i^K)^T}{2}
=\frac{q_a\left(W_i^Q(W_i^K)^T\right)k_b^T}{2}.$$
Here $(8\times4)(4\times8)=8\times8$ for the head-specific compatibility matrix; the complete score product has dimensions $(1\times8)(8\times8)(8\times1)=1\times1$. Because $W_1^Q(W_1^K)^T$ and $W_2^Q(W_2^K)^T$ can differ, the heads can assign different weights to the same token pair.

For example, one head may learn local syntax and another a pronoun–noun relationship. Separate parameters permit specialization but do not guarantee distinct or human-interpretable roles.

### Task 20: Complete formulation of each head
For $i\in\{1,2\}$, the three independent learned projections are $W_i^Q,W_i^K,W_i^V\in\mathbb R^{8\times4}$. Multiplication maps each eight-feature token representation into the four-feature space of that head.

$$Q_i=Q_bW_i^Q,\quad K_i=K_bW_i^K,\quad V_i=V_bW_i^V,
\qquad(n\times8)(8\times4)=n\times4.$$
$$S_i=\frac{Q_iK_i^T}{\sqrt4}=\frac{Q_iK_i^T}{2},
\qquad(n\times4)(4\times n)=n\times n.$$
$$A_i=\operatorname{softmax}_{\rm row}(S_i)\in\mathbb R^{n\times n},\qquad
\operatorname{head}_i=A_iV_i\in\mathbb R^{n\times4}.$$
The final multiplication in each head has dimensions $(n\times n)(n\times4)=n\times4$. Explicitly, the two heads are:

$$\operatorname{head}_1=\operatorname{softmax}_{\rm row}\left(\frac{(Q_bW_1^Q)(K_bW_1^K)^T}{2}\right)(V_bW_1^V),$$
$$\operatorname{head}_2=\operatorname{softmax}_{\rm row}\left(\frac{(Q_bW_2^Q)(K_bW_2^K)^T}{2}\right)(V_bW_2^V).$$
For self-attention, $Q_b=K_b=V_b=X_b\in\mathbb R^{n\times8}$ **before** these head projections. The projected $Q_i$, $K_i$, and $V_i$ can still differ because their learned weights differ.

### Task 21: Combine both heads
Concatenation places the head outputs side by side along the feature axis; it does not add them or increase the number of tokens. A learned output projection mixes the information across heads.

$$H=\operatorname{Concat}(\operatorname{head}_1,\operatorname{head}_2)\in\mathbb R^{n\times(4+4)}=\mathbb R^{n\times8},$$
$$W^O\in\mathbb R^{8\times8},\qquad
\boxed{\operatorname{MultiHead}(Q_b,K_b,V_b)=HW^O\in\mathbb R^{n\times8}}.$$
$$\underbrace{H}_{n\times8}\underbrace{W^O}_{8\times8}=\underbrace{\operatorname{MultiHead}(Q_b,K_b,V_b)}_{n\times8}.$$

| Matrix | Meaning | Dimensions |
|---|---|---|
| $Q_b,K_b,V_b$ | Inputs to the multi-head layer | $n\times8$ each |
| $W_i^Q,W_i^K,W_i^V$ | Learned projections for head $i$ | $8\times4$ each |
| $Q_i,K_i,V_i$ | Projected head representations | $n\times4$ each |
| $K_i^T$ | Transposed head Keys | $4\times n$ |
| $S_i,A_i$ | Scaled scores and row-normalized weights | $n\times n$ each |
| $\operatorname{head}_i$ | Weighted Values for head $i$ | $n\times4$ |
| $H$ | Two concatenated head outputs | $n\times8$ |
| $W^O$ | Learned output projection | $8\times8$ |
| $HW^O$ | Multi-head output | $n\times8$ |

### Small NumPy check of the bonus dimensions
The PDF does not supply numerical eight-dimensional embeddings or head weights. The following seeded random matrices are **illustrative only**, not additional given data or learned weights; the required bonus answer is the derivation above.

In [20]:
rng = np.random.default_rng(42)
X_bonus = rng.normal(size=(3, d_model))
W_Q_heads = rng.normal(size=(h, d_model, head_dimension))
W_K_heads = rng.normal(size=(h, d_model, head_dimension))
W_V_heads = rng.normal(size=(h, d_model, head_dimension))
W_O = rng.normal(size=(d_model, d_model))
print("Bonus input shape:", X_bonus.shape)

Bonus input shape: (3, 8)


In [21]:
heads = []
head_weights = []
for i in range(h):
    Q_i = X_bonus @ W_Q_heads[i]
    K_i = X_bonus @ W_K_heads[i]
    V_i = X_bonus @ W_V_heads[i]
    A_i = softmax_rows((Q_i @ K_i.T) / np.sqrt(head_dimension))
    head_i = A_i @ V_i
    heads.append(head_i)
    head_weights.append(A_i)
    print(f"Head {i + 1}: Q={Q_i.shape}, K={K_i.shape}, V={V_i.shape}, A={A_i.shape}, output={head_i.shape}")

Head 1: Q=(3, 4), K=(3, 4), V=(3, 4), A=(3, 3), output=(3, 4)
Head 2: Q=(3, 4), K=(3, 4), V=(3, 4), A=(3, 3), output=(3, 4)


In [22]:
concatenated = np.concatenate(heads, axis=1)
multihead_output = concatenated @ W_O
print("Concatenated shape:", concatenated.shape)
print("W_O shape:", W_O.shape)
print("Multi-head output shape:", multihead_output.shape)
assert multihead_output.shape == (3, 8)
assert all(np.allclose(weights.sum(axis=1), 1) for weights in head_weights)

Concatenated shape: (3, 8)
W_O shape: (8, 8)
Multi-head output shape: (3, 8)


## Final results and dimension checklist
The three-token example is complete with $d_k=2$:

$$\boxed{A\approx\begin{bmatrix}0.401112&0.197776&0.401112\\0.197776&0.401112&0.401112\\0.248255&0.248255&0.503490\end{bmatrix}},\qquad
\boxed{AV\approx\begin{bmatrix}1.604448&1.197776\\1.197776&1.604448\\1.503490&1.503490\end{bmatrix}}.$$

| Stage | Dimension calculation | Result shape |
|---|---|---|
| $Q=XW_Q$ | $(3\times2)(2\times2)$ | $3\times2$ |
| $K=XW_K$ | $(3\times2)(2\times2)$ | $3\times2$ |
| $V=XW_V$ | $(3\times2)(2\times2)$ | $3\times2$ |
| $K^T$ | Transpose of $3\times2$ | $2\times3$ |
| $QK^T$ | $(3\times2)(2\times3)$ | $3\times3$ |
| $S=QK^T/\sqrt2$ | Matrix divided by a scalar | $3\times3$ |
| $A=\operatorname{softmax}_{\rm row}(S)$ | Normalize each row across Keys | $3\times3$ |
| $O=AV$ | $(3\times3)(3\times2)$ | $3\times2$ |

| Assignment requirement | Covered in |
|---|---|
| Q1, tasks 1–4: $Q,K,V$, every multiplication entry, dimensions | Part A |
| Q2, tasks 5–9: transpose, nine dot products, $d_k$, scaling | Part B |
| Q3, tasks 10–13: exponentials, denominator, three probabilities, sum | Part C |
| Q4: mathematical explanation of normalized attention weights | Part C |
| Full $A$ and $AV$ calculation, including all intermediate matrices | Part D |
| Q5, tasks 14–17: variance proof, scaling, softmax saturation | Part E |
| Q6: Query, Keys, ideal Key, Value, and Transformer connection | Part F |
| Q7, tasks 18–21: two four-dimensional heads and full formulation | Bonus |
| Clearly typeset derivations, short explanations, dimensions, final matrices | Throughout |

All required calculations appear in this notebook. No separate PDF, dataset, or trained model is needed.